In [1]:
import pandas as pd
import numpy as np

## Data Cleaning

In [2]:
# Load dataset
data = pd.read_csv('All_Trimesters.csv')  # Replace with actual path

In [3]:
T1 = data[data['Trimester']=='first']
T2 = data[data['Trimester']=='second']
T3 = data[data['Trimester']=='third']

In [ ]:
## Global Stratified Train-Test Split (So that there is no ORIG_ID leaks between train and test across all datasets)


In [ ]:
# Collect unique ORIG_ID and PTB_NEW mapping
id_label_df = pd.concat([
    df[["ORIG_ID", "PTB_NEW"]].drop_duplicates()
    for df in [T1, T2, T3]
]).drop_duplicates()


In [ ]:
# Stratified split on ORIG_ID
from sklearn.model_selection import train_test_split
train_ids, test_ids = train_test_split(
    id_label_df["ORIG_ID"],
    test_size=0.2,
    stratify=id_label_df["PTB_NEW"],
    random_state=42
)

In [ ]:
# Split each trimester dataset into train and test
T1_train = T1[T1["ORIG_ID"].isin(train_ids)].reset_index(drop=True)
T1_test = T1[T1["ORIG_ID"].isin(test_ids)].reset_index(drop=True)
T2_train = T2[T2["ORIG_ID"].isin(train_ids)].reset_index(drop=True)
T2_test = T2[T2["ORIG_ID"].isin(test_ids)].reset_index(drop=True)
T3_train = T3[T3["ORIG_ID"].isin(train_ids)].reset_index(drop=True)
T3_test = T3[T3["ORIG_ID"].isin(test_ids)].reset_index(drop=True)

In [ ]:
def filter_features_by_nulls(X: pd.DataFrame, y: pd.Series, threshold: float = 0.2) -> pd.DataFrame:
    """
    Keep only features with < threshold fraction of nulls in each target class.
    """
    valid_features = []
    for col in X.columns:
        keep = True
        for cls in y.unique():
            mask = y == cls
            null_frac = X.loc[mask, col].isna().mean()
            if null_frac >= threshold:
                keep = False
                break
        if keep:
            valid_features.append(col)
    
    return X[valid_features]


In [ ]:
# Filter features by nulls on train data, then apply same filter to test
T1_train_filtered = filter_features_by_nulls(T1_train, T1_train['PTB_NEW'])
T2_train_filtered = filter_features_by_nulls(T2_train, T2_train['PTB_NEW'])
T3_train_filtered = filter_features_by_nulls(T3_train, T3_train['PTB_NEW'])

# Get common columns from train data
common_cols = T1_train_filtered.columns.intersection(T2_train_filtered.columns).intersection(T3_train_filtered.columns)

# Apply same column filter to test data
T1_test_filtered = T1_test[common_cols]
T2_test_filtered = T2_test[common_cols]
T3_test_filtered = T3_test[common_cols]

# Apply same column filter to train data
T1_train_common = T1_train_filtered[common_cols]
T2_train_common = T2_train_filtered[common_cols]
T3_train_common = T3_train_filtered[common_cols]

## Data Imputation

In [ ]:
def class_conditional_median_impute(X_train: pd.DataFrame, y_train: pd.Series, 
                                     X_test: pd.DataFrame, y_test: pd.Series) -> tuple:
    """
    Impute NaNs in numeric columns using class-conditional medians from training data.
    Fits on train, applies to both train and test.
    String/categorical columns are left unchanged.
    Returns (X_train_imputed, X_test_imputed)
    """
    # Separate numeric and non-numeric columns
    num_cols = X_train.select_dtypes(include=[np.number]).columns
    non_num_cols = X_train.columns.difference(num_cols)

    X_train_num = X_train[num_cols]
    X_test_num = X_test[num_cols]
    X_train_non_num = X_train[non_num_cols]
    X_test_non_num = X_test[non_num_cols]

    # Convert to numpy
    X_train_arr = X_train_num.to_numpy(dtype=float)
    X_test_arr = X_test_num.to_numpy(dtype=float)
    y_train_arr = y_train.to_numpy()
    y_test_arr = y_test.to_numpy()
    
    # Calculate medians from training data only
    medians_by_class = {}
    for cls in np.unique(y_train_arr):
        mask = (y_train_arr == cls)
        medians_by_class[cls] = np.nanmedian(X_train_arr[mask], axis=0)
    
    # Impute training data
    X_train_imputed = X_train_arr.copy()
    for cls in np.unique(y_train_arr):
        mask = (y_train_arr == cls)
        row_idx = np.where(mask)[0]
        for j in range(X_train_arr.shape[1]):
            col_vals = X_train_imputed[row_idx, j]
            nan_rows = np.isnan(col_vals)
            if np.any(nan_rows):
                X_train_imputed[row_idx[nan_rows], j] = medians_by_class[cls][j]
    
    # Impute test data using training medians
    X_test_imputed = X_test_arr.copy()
    for cls in np.unique(y_test_arr):
        # Use training median for this class, or fallback to overall median if class not in train
        if cls in medians_by_class:
            median_vals = medians_by_class[cls]
        else:
            # Fallback: use overall median from training data
            median_vals = np.nanmedian(X_train_arr, axis=0)
        
        mask = (y_test_arr == cls)
        row_idx = np.where(mask)[0]
        for j in range(X_test_arr.shape[1]):
            col_vals = X_test_imputed[row_idx, j]
            nan_rows = np.isnan(col_vals)
            if np.any(nan_rows):
                X_test_imputed[row_idx[nan_rows], j] = median_vals[j]
    
    # Reconstruct DataFrames
    X_train_imputed_df = pd.DataFrame(X_train_imputed, columns=num_cols, index=X_train.index)
    X_test_imputed_df = pd.DataFrame(X_test_imputed, columns=num_cols, index=X_test.index)
    
    X_train_final = pd.concat([X_train_imputed_df, X_train_non_num], axis=1)[X_train.columns]
    X_test_final = pd.concat([X_test_imputed_df, X_test_non_num], axis=1)[X_test.columns]
    
    return X_train_final, X_test_final

In [ ]:
# Impute missing values (fit on train, apply to test)
T1_train_imputed, T1_test_imputed = class_conditional_median_impute(
    T1_train_common, T1_train_common['PTB_NEW'],
    T1_test_filtered, T1_test_filtered['PTB_NEW']
)
T2_train_imputed, T2_test_imputed = class_conditional_median_impute(
    T2_train_common, T2_train_common['PTB_NEW'],
    T2_test_filtered, T2_test_filtered['PTB_NEW']
)
T3_train_imputed, T3_test_imputed = class_conditional_median_impute(
    T3_train_common, T3_train_common['PTB_NEW'],
    T3_test_filtered, T3_test_filtered['PTB_NEW']
)

In [ ]:
# Save all datasets
datasets_to_save = {
    "T1": (T1_train_imputed, T1_test_imputed),
    "T2": (T2_train_imputed, T2_test_imputed),
    "T3": (T3_train_imputed, T3_test_imputed)
}

for name, (train_df, test_df) in datasets_to_save.items():
    train_df.to_csv(f"{name}_train.csv", index=False)
    test_df.to_csv(f"{name}_test.csv", index=False)
    print(f"Saved {name}_train.csv ({train_df.shape}) and {name}_test.csv ({test_df.shape})")

Saved T1_train.csv ((667, 29596)) and T1_test.csv ((151, 29596))
Saved T2_train.csv ((1303, 29596)) and T2_test.csv ((338, 29596))
Saved T3_train.csv ((838, 29596)) and T3_test.csv ((218, 29596))
Saved T1_T2_train.csv ((240, 118370)) and T1_T2_test.csv ((58, 118370))
Saved T1_T3_train.csv ((345, 118370)) and T1_T3_test.csv ((74, 118370))
Saved T2_T3_train.csv ((425, 118370)) and T2_T3_test.csv ((130, 118370))
